# API_key

In [1]:
import os
os.environ["OPENAI_API_KEY"] = ""

# Read VTT

In [2]:
import webvtt

def load_vtt(file_path):
    lines = []
    for caption in webvtt.read(file_path):
        text = caption.text.strip()
        if text:
            lines.append(text)
    return "\n".join(lines)

transcript = load_vtt("Spotkanie_3.vtt")

# Prompt

In [3]:
PROMPT = """
Na podstawie transkrypcji przygotuj profesjonalną notatkę ze spotkania.

Zasady:
- usuń powtórzenia i przerywniki typu "yyy", "eee"
- popraw język, składnię i interpunkcję
- nie zmieniaj sensu wypowiedzi
- jeśli nie da się czegoś ustalić, wpisz "brak"
- zwróć czysty, formalny tekst po polsku
- NIE używaj emoji ani ikon
- sekcje mają być dokładnie w tej kolejności

Format odpowiedzi:

EXECUTIVE SUMMARY (STRONA 0)

Informacje podstawowe
Data:
Typ:
Obszar:
Status:

Uczestnicy
- Imię Nazwisko – rola

Cel spotkania

Kluczowe ustalenia (TL;DR)
- punkt
- punkt

ZADANIA / ACTION ITEMS
A1 | zadanie | owner | status
A2 | zadanie | owner | status
A3 | zadanie | owner | status

Otwarte tematy
- punkt

NOTATKA SZCZEGÓŁOWA

1. Kontekst

2. Główne wątki / dyskusja

3. Decyzje

4. Kolejne kroki

Notatka sporządzona przez:
Data utworzenia:
"""

In [4]:
from openai import OpenAI

client = OpenAI()

clean_response = client.responses.create(
    model="gpt-5-mini",
    instructions="""
    Oczyść transkrypcję po polsku:
    - usuń przerywniki
    - popraw literówki
    - popraw interpunkcję
    - nie skracaj
    - nie zmieniaj sensu
    """,
    input=transcript
)

clean_text = clean_response.output_text

# tokeny
usage = clean_response.usage
input_tokens = usage.input_tokens
output_tokens = usage.output_tokens

print("Input tokens:", input_tokens)
print("Output tokens:", output_tokens)

# 👇 PRZYKŁADOWE ceny (musisz podstawić aktualne z cennika!)
PRICE_INPUT = 0.000005   # USD za token (przykład!)
PRICE_OUTPUT = 0.000015  # USD za token (przykład!)

cost = input_tokens * PRICE_INPUT + output_tokens * PRICE_OUTPUT

print(f"Koszt zapytania_1: ${cost:.6f}")

response = client.responses.create(
    model="gpt-5-mini",
    instructions=PROMPT,
    input=clean_text
)

note = response.output_text

# tokeny
usage = response.usage
input_tokens = usage.input_tokens
output_tokens = usage.output_tokens

print("Input tokens:", input_tokens)
print("Output tokens:", output_tokens)

# 👇 Ceny
PRICE_INPUT = 0.000005   # USD za token
PRICE_OUTPUT = 0.000015  # USD za token

cost = input_tokens * PRICE_INPUT + output_tokens * PRICE_OUTPUT

print(f"Koszt zapytania_2: ${cost:.6f}")

Input tokens: 4592
Output tokens: 11856
Koszt zapytania_1: $0.200800
Input tokens: 2835
Output tokens: 3187
Koszt zapytania_2: $0.061980


# Save to PDF

In [5]:
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase.pdfmetrics import registerFontFamily
from xml.sax.saxutils import escape
import os
import re
from datetime import datetime


# -------------------------------------------------
# 1. REJESTRACJA FONTU Z POLSKIMI ZNAKAMI
# -------------------------------------------------
def register_fonts():
    candidates = [
        {
            "regular": "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
            "bold": "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
            "italic": "/usr/share/fonts/truetype/dejavu/DejaVuSans-Oblique.ttf",
            "bold_italic": "/usr/share/fonts/truetype/dejavu/DejaVuSans-BoldOblique.ttf",
        },
        {
            "regular": "C:/Windows/Fonts/arial.ttf",
            "bold": "C:/Windows/Fonts/arialbd.ttf",
            "italic": "C:/Windows/Fonts/ariali.ttf",
            "bold_italic": "C:/Windows/Fonts/arialbi.ttf",
        },
    ]

    for cand in candidates:
        if all(os.path.exists(path) for path in cand.values()):
            pdfmetrics.registerFont(TTFont("CustomSans", cand["regular"]))
            pdfmetrics.registerFont(TTFont("CustomSans-Bold", cand["bold"]))
            pdfmetrics.registerFont(TTFont("CustomSans-Italic", cand["italic"]))
            pdfmetrics.registerFont(TTFont("CustomSans-BoldItalic", cand["bold_italic"]))

            registerFontFamily(
                "CustomSans",
                normal="CustomSans",
                bold="CustomSans-Bold",
                italic="CustomSans-Italic",
                boldItalic="CustomSans-BoldItalic",
            )
            return

    raise FileNotFoundError(
        "Nie znaleziono fontu TTF. Podaj własną ścieżkę do fontów TrueType."
    )


# -------------------------------------------------
# 2. STYLE DOKUMENTU
# -------------------------------------------------
def build_styles():
    styles = getSampleStyleSheet()

    styles.add(ParagraphStyle(
        name="DocTitle",
        fontName="CustomSans-Bold",
        fontSize=18,
        leading=24,
        spaceAfter=14,
        alignment=TA_CENTER,
        textColor=colors.HexColor("#1F3A5F"),
    ))

    styles.add(ParagraphStyle(
        name="SectionTitle",
        fontName="CustomSans-Bold",
        fontSize=13,
        leading=16,
        spaceBefore=10,
        spaceAfter=8,
        textColor=colors.HexColor("#17324D"),
    ))

    styles.add(ParagraphStyle(
        name="SubTitle",
        fontName="CustomSans-Bold",
        fontSize=11,
        leading=14,
        spaceBefore=8,
        spaceAfter=6,
        textColor=colors.HexColor("#2F4858"),
    ))

    styles.add(ParagraphStyle(
        name="Body",
        fontName="CustomSans",
        fontSize=10.5,
        leading=15,
        spaceAfter=5,
        alignment=TA_LEFT,
    ))

    styles.add(ParagraphStyle(
        name="Small",
        fontName="CustomSans",
        fontSize=9,
        leading=12,
        textColor=colors.HexColor("#555555"),
        spaceBefore=8,
    ))

    styles.add(ParagraphStyle(
        name="CustomBullet",
        fontName="CustomSans",
        fontSize=10.5,
        leading=15,
        leftIndent=14,
        firstLineIndent=-6,
        spaceAfter=3,
    ))

    styles.add(ParagraphStyle(
        name="TableCell",
        fontName="CustomSans",
        fontSize=9,
        leading=11,
        spaceAfter=0,
        alignment=TA_LEFT,
    ))

    styles.add(ParagraphStyle(
        name="TableHeader",
        fontName="CustomSans-Bold",
        fontSize=9.5,
        leading=11.5,
        spaceAfter=0,
        alignment=TA_LEFT,
        textColor=colors.HexColor("#17324D"),
    ))

    return styles


# -------------------------------------------------
# 3. POMOCNICZE
# -------------------------------------------------
def clean_line(line: str) -> str:
    line = line.strip()
    line = line.replace("📌", "").replace("🗓", "").replace("👥", "")
    line = line.replace("🎯", "").replace("🧠", "").replace("✅", "")
    line = line.replace("🔁", "").replace("📄", "")
    line = line.replace("1️⃣", "1.").replace("2️⃣", "2.")
    line = line.replace("3️⃣", "3.").replace("4️⃣", "4.")
    return line.strip()


def is_section_title(line: str) -> bool:
    normalized = clean_line(line).upper()
    starts = [
        "EXECUTIVE SUMMARY",
        "INFORMACJE PODSTAWOWE",
        "UCZESTNICY",
        "CEL SPOTKANIA",
        "KLUCZOWE USTALENIA",
        "ZADANIA / ACTION ITEMS",
        "OTWARTE TEMATY",
        "NOTATKA SZCZEGÓŁOWA",
        "1. KONTEKST",
        "2. GŁÓWNE WĄTKI / DYSKUSJA",
        "3. DECYZJE",
        "4. KOLEJNE KROKI",
    ]
    return any(normalized.startswith(s) for s in starts)


def is_bullet(line: str) -> bool:
    return line.strip().startswith("- ")


def is_action_row(line: str) -> bool:
    return bool(re.match(r"^A\d+\s*\|", line.strip()))


def format_multiline_cell(text: str) -> str:
    text = escape(text.strip())
    text = text.replace(", ", ",<br/>")
    return text


def build_action_table(action_lines, styles):
    data = [
        [
            Paragraph("ID", styles["TableHeader"]),
            Paragraph("Zadanie", styles["TableHeader"]),
            Paragraph("Owner", styles["TableHeader"]),
            Paragraph("Status", styles["TableHeader"]),
        ]
    ]

    for line in action_lines:
        parts = [p.strip() for p in line.split("|")]
        while len(parts) < 4:
            parts.append("")

        data.append([
            Paragraph(escape(parts[0]), styles["TableCell"]),
            Paragraph(escape(parts[1]), styles["TableCell"]),
            Paragraph(format_multiline_cell(parts[2]), styles["TableCell"]),
            Paragraph(format_multiline_cell(parts[3]), styles["TableCell"]),
        ])

    table = Table(
        data,
        colWidths=[1.4 * cm, 8.8 * cm, 3.5 * cm, 4.0 * cm],
        repeatRows=1,
    )

    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#DCE6F1")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.HexColor("#17324D")),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#9FB3C8")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("ALIGN", (0, 0), (0, -1), "LEFT"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F7FAFC")]),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))

    return table


# -------------------------------------------------
# 4. PDF
# -------------------------------------------------
def add_page_number(canvas, doc):
    canvas.saveState()
    canvas.setFont("CustomSans", 9)
    canvas.setFillColor(colors.grey)
    canvas.drawRightString(19.2 * cm, 1.2 * cm, f"Strona {doc.page}")
    canvas.restoreState()


def save_meeting_note_to_pdf(text: str, filename: str = "notatka_spotkanie.pdf"):
    register_fonts()
    styles = build_styles()

    doc = SimpleDocTemplate(
        filename,
        pagesize=A4,
        leftMargin=2.0 * cm,
        rightMargin=2.0 * cm,
        topMargin=1.8 * cm,
        bottomMargin=1.8 * cm,
        title="Notatka ze spotkania",
        author="OpenAI",
    )

    story = []

    story.append(Paragraph("NOTATKA ZE SPOTKANIA", styles["DocTitle"]))
    story.append(Spacer(1, 0.2 * cm))

    raw_lines = [line.rstrip() for line in text.splitlines()]
    i = 0

    while i < len(raw_lines):
        line = clean_line(raw_lines[i])

        if not line:
            i += 1
            continue

        if is_section_title(line):
            story.append(Spacer(1, 0.15 * cm))
            story.append(Paragraph(escape(line), styles["SectionTitle"]))
            i += 1
            continue

        if line.upper().startswith("ID") and i + 1 < len(raw_lines):
            i += 1
            continue

        if is_action_row(line):
            action_lines = []
            while i < len(raw_lines) and is_action_row(clean_line(raw_lines[i])):
                action_lines.append(clean_line(raw_lines[i]))
                i += 1

            if action_lines:
                table = build_action_table(action_lines, styles)
                story.append(Spacer(1, 0.1 * cm))
                story.append(table)
                story.append(Spacer(1, 0.25 * cm))
            continue

        if is_bullet(line):
            bullet_text = escape(line[2:].strip())
            story.append(Paragraph(f"• {bullet_text}", styles["CustomBullet"]))
            i += 1
            continue

        story.append(Paragraph(escape(line), styles["Body"]))
        i += 1

    story.append(Spacer(1, 0.5 * cm))
    story.append(Paragraph(
        f"Data utworzenia PDF: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        styles["Small"]
    ))

    doc.build(
        story,
        onFirstPage=add_page_number,
        onLaterPages=add_page_number
    )

    print(f"Zapisano PDF: {filename}")

In [6]:
save_meeting_note_to_pdf(note, "notatka_spotkanie.pdf")

Zapisano PDF: notatka_spotkanie.pdf
